In [71]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [72]:
q("""select * from circles;""")

,circle_id,circle_name,region
0,1,Delhi,North
1,2,Punjab,North
2,3,Haryana,North
3,4,Himachal Pradesh,North
4,5,Jammu & Kashmir,North
5,6,Rajasthan,North
6,7,Uttar Pradesh (East),North
7,8,Uttar Pradesh (West),North
8,9,Mumbai,West
9,10,Maharashtra,West


In [73]:
q("""
select count(*) as total_count,
COUNT(DISTINCT circle_id) as distinct_count
FROM circles;




""")

,total_count,distinct_count
0,22,22


In [74]:
q("""
select count(*) as total_count,
COUNT(DISTINCT plan_id) as distinct_count
FROM plans;




""")

,total_count,distinct_count
0,14,14


In [75]:
q("""
select count(*) as total_count,
COUNT(DISTINCT usage_id) as distinct_count
FROM usage_daily;




""")

,total_count,distinct_count
0,1326739,1326739


In [76]:
q("""
select count(*) as total_count,
COUNT(DISTINCT recharge_id) as distinct_count
FROM recharges;




""")

,total_count,distinct_count
0,408089,408089


In [77]:
q("""
select count(*) as total_count,
COUNT(DISTINCT complaint_id) as distinct_count
FROM complaints;




""")

,total_count,distinct_count
0,26841,26841


In [78]:
q("""
SELECT COUNT(*)
FROM circles AS c
JOIN customers AS cu
ON c.circle_id = cu.circle_id;
""")

,COUNT(*)
0,22000


In [79]:
q("""
SELECT COUNT(*)
FROM customers AS cu
JOIN plans AS p
ON cu.current_plan_id = p.plan_id;
""")

,COUNT(*)
0,22000


In [80]:
q("""
SELECT COUNT(*)
FROM customers AS cu
JOIN usage_daily AS u
ON cu.customer_id = u.customer_id;
""")

,COUNT(*)
0,1326739


In [81]:
q("""
SELECT COUNT(*)
FROM customers AS cu
JOIN recharges AS r
ON cu.customer_id = r.customer_id;
""")

,COUNT(*)
0,408089


In [82]:
q("""
SELECT COUNT(*)
FROM customers AS cu
JOIN complaints AS co
ON cu.customer_id = co.customer_id;
""")

,COUNT(*)
0,26841


In [83]:
q("""
SELECT COUNT(*)
FROM recharges AS r
JOIN plans AS p
ON r.plan_id = p.plan_id;
""")

,COUNT(*)
0,408089


In [84]:
q("""
SELECT COUNT(*)
FROM circles AS c
LEFT JOIN customers AS cu
ON c.circle_id = cu.circle_id;
""")

,COUNT(*)
0,22000


In [85]:
q("""
SELECT COUNT(*)
FROM customers AS cu
LEFT JOIN plans AS p
ON cu.current_plan_id = p.plan_id;
""")

,COUNT(*)
0,22000


In [86]:
q("""
SELECT COUNT(*)
FROM customers AS cu
LEFT JOIN usage_daily AS u
ON cu.customer_id = u.customer_id;
""")

,COUNT(*)
0,1326743


In [87]:
q("""
SELECT COUNT(*)
FROM customers AS cu
LEFT JOIN recharges AS r
ON cu.customer_id = r.customer_id;
""")

,COUNT(*)
0,408089


In [88]:
q("""
SELECT COUNT(*)
FROM customers AS cu
LEFT JOIN complaints AS co
ON cu.customer_id = co.customer_id;
""")

,COUNT(*)
0,33472


In [89]:
q("""
SELECT COUNT(*)
FROM recharges AS r
LEFT JOIN plans AS p
ON r.plan_id = p.plan_id;
""")

,COUNT(*)
0,408089


In [90]:
q("""select sum(is_churned=1 and churn_date is not null)
            as left_dated,
        sum(is_churned = 0 and churn_date is null)
        asstayed_blank
from customers;""")

,left_dated,asstayed_blank
0,5226,16774


In [93]:

q("""
SELECT
    c.customer_id,
    c.connection_type,
    (
        SELECT COUNT(*)
        FROM recharges r
        WHERE r.customer_id = c.customer_id
    ) AS payments
FROM customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM usage_daily u
    WHERE u.customer_id = c.customer_id
);
""")


,customer_id,connection_type,payments
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [95]:
q("""WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;""")

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0


In [96]:
q("""SELECT 'usage_daily.usage_date' AS col,        
       MIN(usage_date) AS first_,                
       MAX(usage_date) AS last_                
FROM   usage_daily                    
UNION ALL SELECT 'customers.activation_date',   
       MIN(activation_date), MAX(activation_date)
FROM   customers                              
UNION ALL SELECT 'customers.churn_date',
       MIN(churn_date), MAX(churn_date) 
FROM   customers                      
UNION ALL SELECT 'recharges.recharge_date', 
       MIN(recharge_date), MAX(recharge_date)
FROM   recharges                           
UNION ALL SELECT 'complaints.complaint_date',
       MIN(complaint_date), MAX(complaint_date)
FROM   complaints;""")

,col,first_,last_
0,usage_daily.usage_date,2025-01-01,2025-06-30
1,customers.activation_date,2022-01-01,2025-02-28
2,customers.churn_date,2025-01-16,2025-06-23
3,recharges.recharge_date,2022-01-01,2025-06-27
4,complaints.complaint_date,2025-01-01,2025-06-30


In [97]:
q("""WITH t AS (SELECT customer_id,                
             SUM(data_mb) AS total,           
             COUNT(*) AS days_active         
           FROM usage_daily GROUP BY customer_id) 
SELECT ROUND(AVG(total / days_active), 2)      
         AS per_active_day,                    
       ROUND(AVG(total / 181.0), 2)              
         AS per_calendar_day              
FROM   t;""")

,per_active_day,per_calendar_day
0,1404.44,524.15
